In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [3]:
# Load .env
load_dotenv()

# Load API Key
api_key = os.getenv("GOOGLE_API_KEY")


python-dotenv could not parse statement starting at line 2


In [4]:
# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key
)

# Define state
class ChatState(TypedDict):
    message: str
    response: str

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [5]:
# Chatbot node
def chatbot(state: ChatState):
    user_msg = state["message"]

    # Exit conditions
    if user_msg.lower() in ["bye", "goodbye", "exit", "end"]:
        return {"response": "Chat ended. Have a nice day!"}

    # LLM response
    response = llm.invoke(user_msg)

    return {"response": response.content}

In [6]:
# Build graph
graph = StateGraph(ChatState)
graph.add_node("chatbot", chatbot)

graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

In [7]:
# Persistence
memory = MemorySaver()
workflow = graph.compile(checkpointer=memory)

# Thread config
config = {"configurable": {"thread_id": "user1"}}

In [10]:
# Chat loop
while True:
    user_input = input("You: ")

    result = workflow.invoke({"message": user_input}, config=config)
    print("Bot:", result["response"])

    if user_input.lower() in ["bye", "goodbye", "exit", "end"]:
        break

Bot: Hello! How can I help you today?
Bot: The current Prime Minister of India is **Narendra Modi**.

He has been in office since May 2014 and is a member of the Bharatiya Janata Party (BJP).


ValueError: contents are required.